# Исследование надежности заемщиков

## Описание проекта
Заказчик — кредитный отдел банка. Нужно разобраться, влияет ли семейное положение и количество детей клиента на факт погашения кредита в срок. 
Результаты исследования будут учтены при построении модели кредитного скоринга — специальной системы, которая оценивает способность потенциального заёмщика вернуть кредит банку.

## Описание данных

Входные данные от банка — статистика о платёжеспособности клиентов. 
Путь к файлу: /datasets/data.csv

- children — количество детей в семье
- days_employed — общий трудовой стаж в днях
- dob_years — возраст клиента в годах
- education — уровень образования клиента
- education_id — идентификатор уровня образования
- family_status — семейное положение
- family_status_id — идентификатор семейного положения
- gender — пол клиента
- income_type — тип занятости
- debt — имел ли задолженность по возврату кредитов
- total_income — ежемесячный доход
- purpose — цель получения кредита

## План исследования
**Шаг 1.** Загрузка данных и изучение общей информации
- Импортируем библиотеку pandas. Считаем данные из csv-файла в датафрейм и сохраним в переменную data
- Изучим общую информацию о датасете и выведим первые 20 строчек датафрейма data на экран. 

**Шаг 2.** Предобработка данных
- Изучим, есть ли дубликаты в данных. Поищем пропуски: встречаются ли они, в каких столбцах? Можно ли их обработать или оставить как есть?
- Приведем возможные причины появления пропусков в исходных данных. Объясним, почему заполнить пропуски медианным значением — лучшее решение для количественных переменных.
- В данных могут встречаться артефакты (аномалии) — значения, которые не отражают действительность и появились по какой-то ошибке.
- проверим уникальные значения в столбцах
- для каждого типа занятости выведим медианное значение трудового стажа в днях из столбца days_employed. У двух типов (безработные и пенсионеры) получатся аномально большие значения. Исправить такие значения сложно, поэтому оставим их как есть.
- на основании диапазонов ниже добавим столбец total_income_category с категориями:
    - 0–30000 — 'E';
    - 30001–50000 — 'D';
    - 50001–200000 — 'C';
    - 200001–1000000 — 'B';
    - 1000001 и выше — 'A'.
- создадим функцию, которая на основании данных из столбца purpose сформирует новый столбец purpose_category, куда войдут следующие категории:
    - 'операции с автомобилем',
    - 'операции с недвижимостью',
    - 'проведение свадьбы',
    - 'получение образования'.

**Шаг 3.** Исследовательский анализ данных
- Есть ли зависимость между количеством детей и возвратом кредита в срок?
- Есть ли зависимость между семейным положением и возвратом кредита в срок?
- Есть ли зависимость между уровнем дохода и возвратом кредита в срок?
- Как разные цели кредита влияют на его возврат в срок?
- Ответы сопроводим интерпретацией — поясним, о чём именно говорит полученный нами результат

**Шаг 4.** Напишем общий вывод

### Откроем файлы с данными

In [1]:
import pandas as pd

try:
    data = pd.read_csv('/datasets/data.csv')
except:
    data = pd.read_csv('https://code.s3.yandex.net/datasets/data.csv')

Изучим общую информацию о датасете и выведим первые 20 строчек датафрейма data на экран.

In [2]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 21525 entries, 0 to 21524
Data columns (total 12 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   children          21525 non-null  int64  
 1   days_employed     19351 non-null  float64
 2   dob_years         21525 non-null  int64  
 3   education         21525 non-null  object 
 4   education_id      21525 non-null  int64  
 5   family_status     21525 non-null  object 
 6   family_status_id  21525 non-null  int64  
 7   gender            21525 non-null  object 
 8   income_type       21525 non-null  object 
 9   debt              21525 non-null  int64  
 10  total_income      19351 non-null  float64
 11  purpose           21525 non-null  object 
dtypes: float64(2), int64(5), object(5)
memory usage: 2.0+ MB


In [3]:
data.head(20)

,children,days_employed,dob_years,education,education_id,family_status,family_status_id,gender,income_type,debt,total_income,purpose
0,1,-8437.673028,42,высшее,0,женат / замужем,0,F,сотрудник,0,253875.639453,покупка жилья
1,1,-4024.803754,36,среднее,1,женат / замужем,0,F,сотрудник,0,112080.014102,приобретение автомобиля
2,0,-5623.422610,33,Среднее,1,женат / замужем,0,M,сотрудник,0,145885.952297,покупка жилья
3,3,-4124.747207,32,среднее,1,женат / замужем,0,M,сотрудник,0,267628.550329,дополнительное образование
4,0,340266.072047,53,среднее,1,гражданский брак,1,F,пенсионер,0,158616.077870,сыграть свадьбу
5,0,-926.185831,27,высшее,0,гражданский брак,1,M,компаньон,0,255763.565419,покупка жилья
6,0,-2879.202052,43,высшее,0,женат / замужем,0,F,компаньон,0,240525.971920,операции с жильем
7,0,-152.779569,50,СРЕДНЕЕ,1,женат / замужем,0,M,сотрудник,0,135823.934197,образование
8,2,-6929.865299,35,ВЫСШЕЕ,0,гражданский брак,1,F,сотрудник,0,95856.832424,на проведение свадьбы
9,0,-2188.756445,41,среднее,1,женат / замужем,0,M,сотрудник,0,144425.938277,покупка жилья для семьи


По общей информации можно заметить: 
- наличие пропущенных значений в days_employed и total_income
- наличие отрицательных значений в столбце с общим трудовым стажем в днях(days_employed)
- в столбце education есть одни и те же значения, но записанные по-разному

### Предобработка данных

#### Работа с пропусками

Выведим количество пропущенных значений для каждого столбца

In [4]:
data.isna().sum()

children               0
days_employed       2174
dob_years              0
education              0
education_id           0
family_status          0
family_status_id       0
gender                 0
income_type            0
debt                   0
total_income        2174
purpose                0
dtype: int64

Возможные причины появления пропусков в исходных данных

- человеческий фактор: некоторые данные могли быть просто не указаны(например намеренно сокрыты)
- техническая ошибка: могла возникнуть ошибка при выгрузке данных

Среднее значение некорректно характеризует данные, когда в выборке присутствуют выдающиеся значения, поэтому лучше в таких случаях использовать медиану.

Медиана не чувствительна к выбросам, которые смещают среднее относительно медианы. Медиана же продолжает показывать область наибольшей плотности значений. Именно значения медианы будут вероятнее всего ближе к ожидаемым значениям в пропусках.

В двух столбцах есть пропущенные значения. Один из них — days_employed. Пропуски в этом столбце мы обработаем на следующем этапе. Другой столбец с пропущенными значениями — total_income — хранит данные о доходах. На сумму дохода сильнее всего влияет тип занятости, поэтому заполнить пропуски в этом столбце нужно медианным значением по каждому типу из столбца income_type. Например, у человека с типом занятости сотрудник пропуск в столбце total_income должен быть заполнен медианным доходом среди всех записей с тем же типом.

In [5]:
for t in data['income_type'].unique():
    data.loc[(data['income_type'] == t) & (data['total_income'].isna()), 'total_income'] = \
    data.loc[(data['income_type'] == t), 'total_income'].median()

#### Обработка аномальных значений

В данных могут встречаться артефакты (аномалии) — значения, которые не отражают действительность и появились по какой-то ошибке. таким артефактом будет отрицательное количество дней трудового стажа в столбце days_employed. Для реальных данных это нормально. Обработаем значения в этом столбце: заменим все отрицательные значения положительными с помощью метода abs().

In [6]:
data['days_employed'] = data['days_employed'].abs()

Для каждого типа занятости выведем медианное значение трудового стажа days_employed в днях.

In [7]:
data.groupby('income_type')['days_employed'].agg('median')

income_type
безработный        366413.652744
в декрете            3296.759962
госслужащий          2689.368353
компаньон            1547.382223
пенсионер          365213.306266
предприниматель       520.848083
сотрудник            1574.202821
студент               578.751554
Name: days_employed, dtype: float64

У двух типов (безработные и пенсионеры) получатся аномально большие значения. Исправить такие значения сложно, поэтому оставим их как есть.

Выведем перечень уникальных значений столбца children

In [8]:
data['children'].unique()

array([ 1,  0,  3,  2, -1,  4, 20,  5])

 В столбце children есть два аномальных значения. Удалим строки, в которых встречаются такие аномальные значения из датафрейма data.

In [9]:
# Ещё раз выведем перечень уникальных значений столбца children, чтобы убедиться, что артефакты удалены.
data['children'].unique()

array([ 1,  0,  3,  2, -1,  4, 20,  5])

#### Вернемся к пропускам

Заполним пропуски в столбце days_employed медианными значениями по каждого типа занятости income_type.

In [10]:
for t in data['income_type'].unique():
    data.loc[(data['income_type'] == t) & (data['days_employed'].isna()), 'days_employed'] = \
    data.loc[(data['income_type'] == t), 'days_employed'].median()

In [11]:
#Убедимся, что все пропуски заполнены
data.isna().sum()

children            0
days_employed       0
dob_years           0
education           0
education_id        0
family_status       0
family_status_id    0
gender              0
income_type         0
debt                0
total_income        0
purpose             0
dtype: int64

#### Изменение типов данных

Заменим вещественный тип данных в столбце total_income на целочисленный с помощью метода astype().

In [12]:
data['total_income'] = data['total_income'].astype(int)

#### Обработка дубликатов

 Обработаем неявные дубликаты в столбце education. В этом столбце есть одни и те же значения, но записанные по-разному: с использованием заглавных и строчных букв. Приведите их к нижнему регистру.

In [13]:
data['education'] = data['education'].str.lower()

Выведем на экран количество строк-дубликатов в данных. Если такие строки присутствуют, удалим их.

In [14]:
data.duplicated().sum()

71

In [15]:
data = data.drop_duplicates()

#### Категоризация данных

Создадим столбец total_income_category с категориями:

- 0–30000 — 'E';
- 30001–50000 — 'D';
- 50001–200000 — 'C';
- 200001–1000000 — 'B';
- 1000001 и выше — 'A'.

In [16]:
def categorize_income(income):
    try:
        if 0 <= income <= 30000:
            return 'E'
        elif 30001 <= income <= 50000:
            return 'D'
        elif 50001 <= income <= 200000:
            return 'C'
        elif 200001 <= income <= 1000000:
            return 'B'
        elif income >= 1000001:
            return 'A'
    except:
        pass
    
data['total_income_category'] = data['total_income'].apply(categorize_income)
data['total_income_category']

0        B
1        C
2        C
3        B
4        C
        ..
21520    B
21521    C
21522    C
21523    B
21524    C
Name: total_income_category, Length: 21454, dtype: object

Выведем перечень уникальных целей взятия кредита из столбца purpose.

In [17]:
data['purpose'].unique()

array(['покупка жилья', 'приобретение автомобиля',
       'дополнительное образование', 'сыграть свадьбу',
       'операции с жильем', 'образование', 'на проведение свадьбы',
       'покупка жилья для семьи', 'покупка недвижимости',
       'покупка коммерческой недвижимости', 'покупка жилой недвижимости',
       'строительство собственной недвижимости', 'недвижимость',
       'строительство недвижимости', 'на покупку подержанного автомобиля',
       'на покупку своего автомобиля',
       'операции с коммерческой недвижимостью',
       'строительство жилой недвижимости', 'жилье',
       'операции со своей недвижимостью', 'автомобили',
       'заняться образованием', 'сделка с подержанным автомобилем',
       'получение образования', 'автомобиль', 'свадьба',
       'получение дополнительного образования', 'покупка своего жилья',
       'операции с недвижимостью', 'получение высшего образования',
       'свой автомобиль', 'сделка с автомобилем',
       'профильное образование', 'высшее об

 Создадим функцию, которая на основании данных из столбца purpose сформирует новый столбец purpose_category, в который войдут следующие категории:

- 'операции с автомобилем',
- 'операции с недвижимостью',
- 'проведение свадьбы',
- 'получение образования'.

In [18]:
def categorize_purpose(row):
    try:
        if 'автом' in row:
            return 'операции с автомобилем'
        elif 'жил' in row or 'недвиж' in row:
            return 'операции с недвижимостью'
        elif 'свад' in row:
            return 'проведение свадьбы'
        elif 'образов' in row:
            return 'получение образования'
    except:
        return 'нет категории'
    
data['purpose_category'] = data['purpose'].apply(categorize_purpose)
data['purpose_category']

0        операции с недвижимостью
1          операции с автомобилем
2        операции с недвижимостью
3           получение образования
4              проведение свадьбы
                   ...           
21520    операции с недвижимостью
21521      операции с автомобилем
21522    операции с недвижимостью
21523      операции с автомобилем
21524      операции с автомобилем
Name: purpose_category, Length: 21454, dtype: object

Мы избавились от пропусков, дубликатов, аномальных значений, поменяли тип данных, где это было нужно, и добавили новые столбцы, которые впоследствие нам пригодятся для анализа

### Исследовательский анализ данных

Изучим:

1. Наличие зависимости между количеством детей и возвратом кредита в срок

In [19]:
#создаем функцию для перевода результатов в проценты
def percent(x):
    cnt = round(x.mean()*100, 2)
    return str(cnt)+"%"
#создаем сводную таблицу по столбцам с кол-вом детей("children") и количеством кредитов: 
#считаем по задолженности("debt"):
child_pivot = data.pivot_table(index=["children"], values="debt", aggfunc=['count','sum', percent])
#переименовываем столбцы для наглядности
child_pivot.columns = ['sum_child', 'amount_debt', 'share_child']
#выводим таблицу 
child_pivot

,sum_child,amount_debt,share_child
children,,,
-1,47,1,2.13%
0,14091,1063,7.54%
1,4808,444,9.23%
2,2052,194,9.45%
3,330,27,8.18%
4,41,4,9.76%
5,9,0,0.0%
20,76,8,10.53%


Вывод: Информации по семьям с 5 детьми недостаточно,чтобы делать определенные выводы(либо их мало, либо пропуски в данных ставят под сомнение этот результат), также и с 3 и 4 детьми(мало данных), поэтому рассмотрим семьи с 0, 1 и 2 детьми: доля просроченных кредитов от общего числа по группе с детьми около 9,2-9,45%, это выше, чем в группе без детей(7,5%). Можно заметить очевидную зависимость между наличием детей и возвратом кредита в срок

2. Наличие зависимости между семейным положением и возвратом кредита в срок

In [20]:
#создаем сводную таблицу по столбцам с семейным положением("family_status") и количеством кредитов: 
#считаем по задолженности("debt"): 0 - вовремя погашеные кредиты, 1- есть задолженность
fam_pivot = data.pivot_table(index=["family_status"], columns="debt", values="children", aggfunc='count')

#считаем общее количество кредитов и создаем для этих значений отдельный столбец "sum_fam_status":
fam_pivot["sum_fam_status"] = fam_pivot[0] + fam_pivot[1]

#считаем долю просроченных кредитов от общего числа для каждого значения семейного положения
#создаем для нее отдельный столбец "share_fam"
fam_pivot["share_fam"] = (fam_pivot[1] / fam_pivot["sum_fam_status"])*100

#удаляем столбец с вовремя погашенными кредитами 
del fam_pivot[0]
#переименовываем столбец "1", чтобы было понятно содержание столбца
fam_pivot = fam_pivot.rename(columns={1:"amount_debt"})

fam_pivot

debt,amount_debt,sum_fam_status,share_fam
family_status,,,
Не женат / не замужем,274,2810,9.750890
в разводе,85,1195,7.112971
вдовец / вдова,63,959,6.569343
гражданский брак,388,4151,9.347145
женат / замужем,931,12339,7.545182


Вывод: изучая данные можно заметить разницу между семейным положением и возвратом кредита в срок: особенно выделяются группы "женат / замужем", где доля задолженности ~7,6% и "гражданский брак","Не женат / не замужем", где доля задолженности 9,3-9,7%, что значительно выше, чем у других групп данного сегмента. Полученный результат говорит нам о том, что стоит учитывать семейное положение, ведь влияние на факт погашения кредита в срок присутствует.

3. Наличие зависимости между уровнем дохода и возвратом кредита в срок

In [21]:
#создаем сводную таблицу по столбцам с доходом, разделенным на категории("total_income_category") и количеством кредитов: 
#считаем по задолженности("debt"): 0 - вовремя погашеные кредиты, 1- есть задолженность
income_pivot = data.pivot_table(index=['total_income_category'], columns="debt", values="children", aggfunc='count')

#считаем общее количество кредитов и создаем для этих значений отдельный столбец "sum_income":
income_pivot["sum_income"] = income_pivot[0] + income_pivot[1]

#считаем долю просроченных кредитов от общего числа для каждого значения семейного положения
#создаем для нее отдельный столбец "share_inc_category"
income_pivot["share_inc_category"] = (income_pivot[1] / income_pivot["sum_income"])*100

#удаляем столбец с вовремя погашенными кредитами 
del income_pivot[0]
#переименовываем столбец "1", чтобы было понятно содержание столбца
income_pivot = income_pivot.rename(columns={1:"amount_debt"})

income_pivot

debt,amount_debt,sum_income,share_inc_category
total_income_category,,,
A,2,25,8.000000
B,356,5042,7.060690
C,1360,16015,8.492039
D,21,350,6.000000
E,2,22,9.090909


Вывод: в категориях "A","E" мало данных(мало общего числа кредитов), поэтому рассмотрим категории с более наглядными данными: в категории "B" доля задолженности составляет ~7% от общего числа кредитов в этой группе, в категории "C" ~8,5%. Может показаться, что здесь присутствует взаимосвязь доли возврата кредита вовремя с уровнем дохода: чем больше доход, тем выше вероятность возврата кредита в срок. Но если обратить внимание на долю просроченных кредитов в категории "D", то она равна ~6%, хотя эта категория и включает данные с более низким доходом. Что опровергает раннее установленную зависимость вовзврата кредита в срок от уровня дохода

4. Влияние разных целей кредита на его возврат в срок

In [22]:
#создаем сводную таблицу по столбцам с целью получения кредита("purpose_category") и количеством кредитов: 
#считаем по задолженности("debt"): 0 - вовремя погашеные кредиты, 1- есть задолженность
purpose_pivot = data.pivot_table(index=['purpose_category'], columns="debt", values="children", aggfunc='count')

#считаем общее количество кредитов и создаем для этих значений отдельный столбец "sum_purpose":
purpose_pivot["sum_purpose"] = purpose_pivot[0] + purpose_pivot[1]

#считаем долю просроченных кредитов от общего числа для каждого значения цели
#создаем для нее отдельный столбец "share_purpose"
purpose_pivot["share_purpose"] = (purpose_pivot[1] / purpose_pivot["sum_purpose"])*100

#удаляем столбец с вовремя погашенными кредитами 
del purpose_pivot[0]
#переименовываем столбец "1", чтобы было понятно содержание столбца
purpose_pivot = purpose_pivot.rename(columns={1:"amount_debt"})

purpose_pivot

debt,amount_debt,sum_purpose,share_purpose
purpose_category,,,
операции с автомобилем,403,4306,9.359034
операции с недвижимостью,782,10811,7.233373
получение образования,370,4013,9.220035
проведение свадьбы,186,2324,8.003442


Вывод: цели "операции с автомобилем" и "получение образования" имеют более высокую долю просроченных кредитов (9,2-9,3%), чем цели "операции с недвижимостью" и "проведение свадьбы" - 7,3-7,9%, при этом цель "операции с недвижимостью" имеет больше всех данных по кредитам и меньшую долю просроченных, что говорит о высокой вероятности возврата кредита в срок. Такие выводы дают возможность судить о влиянии цели кредита на его возврат в срок

### Общий вывод

- Если рассматривать зависимость возврата кредита в срок от наличия детей: группы с детьми несут больше рисков по невозврату кредита, чем бездетные. Но при этом если обратить внимание на результаты по семейному положению: у категории "гражданский брак","Не женат / не замужем" доля просроченных кредитов больше, чем у группы "женат / замужем".
  
- Далее при рассматрении зависимости доли возврата кредита вовремя от целей кредита, то категории "операции с автомобилем" и "получение образования" имеют более высокую долю просроченных кредитов, поэтому здесь больше рисков, стоит учитывать. 
  
- Что же касается  уровня дохода: сложно установить взаимосвязь, так как закономерности в сравнении результатов не было установлено: при выстраивании категорий от наименьшего к наибольшему, ожидаем, что между целевым признаком (долей невозврата) и доходом может быть линейная зависимость (чем больше один признак, тем больше другой или тем меньше другой). Но подобное не наблюдается
- Поэтому в будущем правильнее будет посмотреть на влияние нескольких факторов одновременно на факт погашения кредита в срок, например тот же доход возможно в сочетании с другими(например с семейным положением) будет хорошим показателем и окажет влияние на возврат в срок.